# FinGPT × MedicalGPT：SFT + DPO 最小可复用训练流程

目标：把 **FinGPT 的金融任务数据（内容）** 映射到 **MedicalGPT 的多阶段训练方法（SFT + DPO）**。

本 Notebook 对齐：
- 数据格式：`docs/datasets.md`
- pipeline 参考：`run_training_dpo_pipeline.ipynb`

最终会产出：
1. FinGPT -> MedicalGPT SFT 格式（ShareGPT conversations）
2. FinGPT -> MedicalGPT DPO 格式（question/chosen/rejected）
3. 通用数据（`data/finetune/sharegpt_zh_1K_format.jsonl`）+ 领域数据（FinGPT）混合 SFT 训练集
4. 基于 Qwen2.5-7B 的 SFT 与 DPO 训练命令


## 0. 环境准备（可选）
如果你在全新环境运行，请先安装依赖；已按项目 README 配好可跳过。


In [ ]:
# HF Mirror（可选）
import os
HF_ENDPOINT = "https://hf-mirror.com"
os.environ["HF_ENDPOINT"] = HF_ENDPOINT
print("HF_ENDPOINT:", os.getenv("HF_ENDPOINT"))


## 1. 配置参数（支持“通用数据 + FinGPT领域数据”）


In [ ]:
from pathlib import Path

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# 1) 通用数据（MedicalGPT内置可复用）
GENERAL_SFT_FILES = [
    Path("data/finetune/sharegpt_zh_1K_format.jsonl"),
    # 可按 docs/datasets.md 增加更多通用SFT数据
]

# 2) 领域数据（FinGPT）
FIN_DATASETS = [
    "FinGPT/fingpt-sentiment-train",  # 示例：情感分析
    # "FinGPT/fingpt-headline",
    # "FinGPT/fingpt-fiqa_qa",
    # "FinGPT/fingpt-finred",
    # "FinGPT/fingpt-ner",
    # "FinGPT/fingpt-fineval",
]
FIN_SPLIT = "train"

OUT_DIR = Path("data/fingpt_medicalgpt")
RAW_DIR = OUT_DIR / "raw"
FIN_SFT_DIR = OUT_DIR / "fin_sft"
FIN_DPO_DIR = OUT_DIR / "fin_dpo"
MIXED_SFT_DIR = OUT_DIR / "mixed_sft"
MERGED_DPO_DIR = OUT_DIR / "merged_dpo"

MIXED_SFT_FILE = MIXED_SFT_DIR / "train_mixed_sft.jsonl"
MERGED_DPO_FILE = MERGED_DPO_DIR / "train_merged_dpo.jsonl"

SFT_OUT = Path("outputs/fingpt_medicalgpt_sft_lora")
DPO_OUT = Path("outputs/fingpt_medicalgpt_dpo_lora")

for d in [RAW_DIR, FIN_SFT_DIR, FIN_DPO_DIR, MIXED_SFT_DIR, MERGED_DPO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_MODEL:", BASE_MODEL)
print("GENERAL_SFT_FILES:", GENERAL_SFT_FILES)
print("FIN_DATASETS:", FIN_DATASETS)
print("MIXED_SFT_FILE:", MIXED_SFT_FILE)
print("MERGED_DPO_FILE:", MERGED_DPO_FILE)


## 2. 下载 FinGPT 领域数据，并转换为 MedicalGPT SFT / DPO 格式


In [ ]:
import json
import subprocess
from datasets import load_dataset


def safe_name(hf_dataset_name: str) -> str:
    # FinGPT/fingpt-sentiment-train -> fingpt-sentiment-train
    return hf_dataset_name.split("/")[-1]

fin_sft_files = []
fin_dpo_files = []

for ds_name in FIN_DATASETS:
    tag = safe_name(ds_name)
    raw_file = RAW_DIR / f"{tag}_{FIN_SPLIT}.jsonl"
    sft_file = FIN_SFT_DIR / f"{tag}_{FIN_SPLIT}_sharegpt.jsonl"
    dpo_file = FIN_DPO_DIR / f"{tag}_{FIN_SPLIT}_dpo.jsonl"

    ds = load_dataset(ds_name, split=FIN_SPLIT)
    with raw_file.open("w", encoding="utf-8") as f:
        for row in ds:
            f.write(json.dumps(dict(row), ensure_ascii=False) + "\n")

    subprocess.run([
        "python", "fin_to_sharegpt.py",
        "--source_file", str(raw_file),
        "--output_file", str(sft_file),
    ], check=True)

    subprocess.run([
        "python", "fin_to_dpo_pairs.py",
        "--source_file", str(raw_file),
        "--output_file", str(dpo_file),
        "--seed", "42",
    ], check=True)

    fin_sft_files.append(sft_file)
    fin_dpo_files.append(dpo_file)
    print(f"[{ds_name}] raw={raw_file} sft={sft_file} dpo={dpo_file}")


## 3. 构建训练数据

- SFT：`通用数据 + FinGPT领域数据` 合并
- DPO：合并 FinGPT 产出的 DPO 数据


In [ ]:
from itertools import islice

# 3.1 合并 SFT
with MIXED_SFT_FILE.open("w", encoding="utf-8") as wf:
    for gfile in GENERAL_SFT_FILES:
        if not gfile.exists():
            print(f"[warn] missing general sft file: {gfile}")
            continue
        with gfile.open("r", encoding="utf-8") as rf:
            for line in rf:
                wf.write(line)
    for ffile in fin_sft_files:
        with ffile.open("r", encoding="utf-8") as rf:
            for line in rf:
                wf.write(line)

# 3.2 合并 DPO
with MERGED_DPO_FILE.open("w", encoding="utf-8") as wf:
    for ffile in fin_dpo_files:
        with ffile.open("r", encoding="utf-8") as rf:
            for line in rf:
                wf.write(line)

print("mixed sft:", MIXED_SFT_FILE)
print("merged dpo:", MERGED_DPO_FILE)

print("\n[SFT sample]")
with MIXED_SFT_FILE.open("r", encoding="utf-8") as f:
    for line in islice(f, 2):
        print(line.strip())

print("\n[DPO sample]")
with MERGED_DPO_FILE.open("r", encoding="utf-8") as f:
    for line in islice(f, 2):
        print(line.strip())


## 4. SFT 训练（MedicalGPT Stage2 思路）

说明：
- `--train_file_dir` 接目录，这里我们把混合后的 SFT 文件放在 `MIXED_SFT_DIR`。
- 可按显存调 `batch size / grad accumulation / max length`。


In [ ]:
import subprocess

sft_cmd = [
    "python", "supervised_finetuning.py",
    "--model_name_or_path", BASE_MODEL,
    "--tokenizer_name_or_path", BASE_MODEL,
    "--train_file_dir", str(MIXED_SFT_DIR),
    "--validation_split_percentage", "1",
    "--do_train",
    "--use_peft", "True",
    "--num_train_epochs", "3",
    "--per_device_train_batch_size", "2",
    "--gradient_accumulation_steps", "8",
    "--learning_rate", "2e-4",
    "--logging_steps", "10",
    "--save_steps", "200",
    "--model_max_length", "1024",
    "--target_modules", "all",
    "--lora_rank", "8",
    "--lora_alpha", "16",
    "--lora_dropout", "0.05",
    "--torch_dtype", "float16",
    "--device_map", "auto",
    "--output_dir", str(SFT_OUT),
    "--overwrite_output_dir",
]
print(" ".join(sft_cmd))
# subprocess.run(sft_cmd, check=True)


## 5. DPO 训练（MedicalGPT Stage3 思路）

说明：
- DPO 起点模型使用 SFT 输出：`--model_name_or_path {SFT_OUT}`
- DPO 训练数据目录使用 `MERGED_DPO_DIR`。


In [ ]:
dpo_cmd = [
    "python", "dpo_training.py",
    "--model_name_or_path", str(SFT_OUT),
    "--tokenizer_name_or_path", BASE_MODEL,
    "--train_file_dir", str(MERGED_DPO_DIR),
    "--validation_split_percentage", "1",
    "--do_train",
    "--use_peft", "True",
    "--per_device_train_batch_size", "2",
    "--gradient_accumulation_steps", "8",
    "--learning_rate", "5e-7",
    "--num_train_epochs", "2",
    "--max_length", "1024",
    "--max_prompt_length", "512",
    "--beta", "0.1",
    "--logging_steps", "10",
    "--save_steps", "200",
    "--target_modules", "all",
    "--lora_rank", "8",
    "--lora_alpha", "16",
    "--lora_dropout", "0.05",
    "--torch_dtype", "float16",
    "--device_map", "auto",
    "--output_dir", str(DPO_OUT),
    "--overwrite_output_dir",
]
print(" ".join(dpo_cmd))
# subprocess.run(dpo_cmd, check=True)


## 6. （可选）合并 LoRA 权重

如需导出可直接推理权重，可对 SFT / DPO 各自执行 merge。


In [ ]:
merge_sft_cmd = [
    "python", "merge_peft_adapter.py",
    "--base_model", BASE_MODEL,
    "--tokenizer_path", BASE_MODEL,
    "--lora_model", str(SFT_OUT),
    "--output_dir", "outputs/fingpt_medicalgpt_sft_merged",
]
print(" ".join(merge_sft_cmd))

merge_dpo_cmd = [
    "python", "merge_peft_adapter.py",
    "--base_model", BASE_MODEL,
    "--tokenizer_path", BASE_MODEL,
    "--lora_model", str(DPO_OUT),
    "--output_dir", "outputs/fingpt_medicalgpt_dpo_merged",
]
print(" ".join(merge_dpo_cmd))
